# Baseline Recommender System

This notebook implements baseline recommendation methods:
1. **Popularity-based**: Recommend most popular posts
2. **Item-based CF**: Cosine similarity between post features

## Data Flow:
1. Load posts and user interaction data
2. Create post features (TF-IDF on captions + descriptions)
3. Implement baseline methods
4. Evaluate using Hit Rate @ 5 and MRR

## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries loaded successfully")

Libraries loaded successfully


## 2. Load Data

In [7]:
# Load data
# interaction_sequences.csv has: commentUser, interaction_sequence, posts_sequence, sequence_length
interactions_df = pd.read_csv('input/interaction_sequences.csv')

# Load posts data for captions
posts_df = pd.read_csv('data/clean/clean_posts.csv')

print(f"Interactions: {interactions_df.shape}")
print(f"Posts: {posts_df.shape}")

# Display sample
print("\n=== Sample Interactions ===")
display(interactions_df.head(3))

print("\n=== Sample Posts ===")
display(posts_df[['post_id', 'caption', 'hashtags','likesCount', 'commentsCount']].head(3))

Interactions: (238, 4)
Posts: (827, 7)

=== Sample Interactions ===


,commentUser,interaction_sequence,posts_sequence,sequence_length
0,92thestudio,"[{'post_id': 188, 'timestamp': '2023-05-17 14:...","[188, 216, 91, 57]",4
1,_surisingh,"[{'post_id': 369, 'timestamp': '2023-05-18 09:...","[369, 361, 360]",3
2,_vanessa.riedl,"[{'post_id': 246, 'timestamp': '2023-05-01 13:...","[246, 88, 57]",3



=== Sample Posts ===


,post_id,caption,hashtags,likesCount,commentsCount
0,1,Cannes 2023 with @kilianparis 🤍 #KilianCannes ...,KilianCannes,32070,142
1,2,"As the clock struck midnight to ring in 2022, ...",no_hashtag,6565,86
2,3,The famous stairs 🤎 Photo @gustave_durin Dre...,no_hashtag,28936,123


In [4]:
# Create post features from captions
posts_df['caption_clean'] = posts_df['caption'].fillna('')

# TF-IDF on captions
tfidf = TfidfVectorizer(max_features=256, ngram_range=(1, 2), min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(posts_df['caption_clean'])
tfidf_features = normalize(tfidf_matrix.toarray(), axis=1)

# Feature mapping: post_id -> embedding
post_features = {}
for idx, post_id in enumerate(posts_df['post_id']):
    post_features[post_id] = tfidf_features[idx]

print(f"TF-IDF features shape: {tfidf_features.shape}")
print(f"Total posts with features: {len(post_features)}")

# Get posts that are in user sequences
posts_in_sequences = set()
for seq in interactions_df['posts_sequence']:
    posts_in_sequences.update(seq)

print(f"Posts in user sequences: {len(posts_in_sequences)}")
print(f"Posts with features: {len(post_features)}")

TF-IDF features shape: (827, 256)
Total posts with features: 827
Posts in user sequences: 14
Posts with features: 827


# Count post popularity from all sequences
from collections import Counter

all_posts = []
for seq in df_sequence['posts_sequence']:
    all_posts.extend(seq)

post_popularity = Counter(all_posts)
most_popular = [pid for pid, _ in post_popularity.most_common(100)]

print(f"Most popular posts (top 10): {most_popular[:10]}")
print(f"Popularity counts: {[post_popularity[pid] for pid in most_popular[:10]]}")

In [5]:
def create_text_features(row):
    """Combine caption, description, and fashion attributes."""
    parts = []
    if pd.notna(row['caption']) and row['caption']:
        parts.append(str(row['caption']))
    if pd.notna(row['description']) and row['description']:
        parts.append(str(row['description']))
    if pd.notna(row['color']) and row['color']:
        parts.append(str(row['color']))
    if pd.notna(row['category']) and row['category']:
        parts.append(str(row['category']))
    if pd.notna(row['style']) and row['style']:
        parts.append(str(row['style']))
    if pd.notna(row['occasion']) and row['occasion']:
        parts.append(str(row['occasion']))
    if pd.notna(row['keywords_posts']) and row['keywords_posts']:
        parts.append(str(row['keywords_posts']))
    return ' '.join(parts)

posts_df['text_features'] = posts_df.apply(create_text_features, axis=1)

# TF-IDF
tfidf = TfidfVectorizer(max_features=256, ngram_range=(1, 2), min_df=2, max_df=0.8)
tfidf_matrix = tfidf.fit_transform(posts_df['text_features'])
tfidf_features = normalize(tfidf_matrix.toarray(), axis=1)

# Feature mapping
post_features = {}
for idx, post_id in enumerate(posts_df['post_id']):
    post_features[post_id] = tfidf_features[idx]

print(f"TF-IDF features shape: {tfidf_features.shape}")

KeyError: 'description'

## 5. Baseline Method 1: Popularity-Based

In [ ]:
# Count post popularity (number of interactions)
post_popularity = Counter(user_interactions['post_id'])
most_popular = [pid for pid, _ in post_popularity.most_common(100)]

print(f"Most popular posts (top 10): {most_popular[:10]}")
print(f"Popularity counts: {[post_popularity[pid] for pid in most_popular[:10]]}")

In [ ]:
def recommend_popularity(user_history, k=5):
    """Recommend most popular posts (not in user history)."""
    recommendations = []
    for post_id in most_popular:
        if post_id not in user_history:
            recommendations.append(post_id)
        if len(recommendations) >= k:
            break
    return recommendations

## 6. Baseline Method 2: Item-Based Collaborative Filtering

In [ ]:
def recommend_item_based_cf(user_history, post_features, k=5):
    """Recommend based on similarity to user's history."""
    if len(user_history) == 0:
        return []
    
    # Get features for user's history
    history_features = [post_features[pid] for pid in user_history if pid in post_features]
    if len(history_features) == 0:
        return []
    
    # Average history features
    user_profile_vector = np.mean(history_features, axis=0)
    
    # All posts and their features
    all_post_ids = list(post_features.keys())
    all_embeddings = np.array([post_features[p] for p in all_post_ids])
    
    # Compute similarity
    similarities = cosine_similarity([user_profile_vector], all_embeddings)[0]
    
    # Get top-k (excluding history)
    recommendations = []
    sorted_indices = np.argsort(similarities)[::-1]
    for idx in sorted_indices:
        post_id = all_post_ids[idx]
        if post_id not in user_history:
            recommendations.append(post_id)
        if len(recommendations) >= k:
            break
    
    return recommendations

print("Item-based CF recommender ready")

## 7. Evaluation

In [ ]:
def evaluate_method(recommend_func, sequences, k=5, method_name="Method"):
    """Evaluate recommendation method."""
    hits = 0
    mrr_sum = 0
    total = 0
    
    for seq in sequences:
        if len(seq) < 2:
            continue
        
        history = seq[:-1]
        ground_truth = seq[-1]
        
        # Get recommendations
        recommendations = recommend_func(history)
        
        # Check hit
        if ground_truth in recommendations:
            hits += 1
            rank = recommendations.index(ground_truth) + 1
            mrr_sum += 1.0 / rank
        
        total += 1
    
    hit_rate = hits / total if total > 0 else 0
    mrr = mrr_sum / total if total > 0 else 0
    
    print(f"\n=== {method_name} ===")
    print(f"Hit Rate @ {k}: {hit_rate:.4f}")
    print(f"MRR: {mrr:.4f}")
    print(f"Total test cases: {total}")
    
    return hit_rate, mrr

In [ ]:
# Get sequences
sequences = df_sequence['posts_sequence'].tolist()

# Evaluate Popularity-based
hr_pop, mrr_pop = evaluate_method(
    recommend_popularity, 
    sequences, 
    k=5, 
    method_name="Popularity-Based"
)

# Evaluate Item-based CF
hr_cf, mrr_cf = evaluate_method(
    lambda history: recommend_item_based_cf(history, post_features, k=5),
    sequences,
    k=5,
    method_name="Item-Based CF"
)

## 8. Summary

In [ ]:
# Summary table
results = pd.DataFrame({
    'Method': ['Popularity-Based', 'Item-Based CF'],
    'Hit Rate @ 5': [hr_pop, hr_cf],
    'MRR': [mrr_pop, mrr_cf]
})

print("\n=== BASELINE RESULTS ===")
display(results)

# Save results
results.to_csv('results/baseline_results.csv', index=False)
print("\nResults saved to results/baseline_results.csv")